In [1]:
# 预处理数据文件

from collections import defaultdict
import os
import pandas as pd

data_path = "../assets/初赛数据/"


def merge_csv_files(input_path, out_path):
    """
    将同一设备不同时间的数据合并为一个文件
    """
    file_groups = defaultdict(list)
    for file_name in os.listdir(input_path):
        if file_name.endswith(".csv") and "字段释义" not in file_name:
            prefix = file_name.rsplit("_", 1)[0]
            file_groups[prefix].append(os.path.join(input_path, file_name))

    for prefix, file_list in file_groups.items():
        merged_df = pd.concat(
            (pd.read_csv(file) for file in file_list), ignore_index=True
        )
        output_file = os.path.join(out_path, f"{prefix}.csv")
        merged_df.rename(columns={merged_df.columns[0]: "index"}, inplace=True)
        print(output_file)
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        merged_df.to_csv(output_file, index=False)
        print(f'Merged files with prefix "{prefix}" into {output_file}')


merge_csv_files(data_path, "data")

data\Ajia_plc_1.csv
Merged files with prefix "Ajia_plc_1" into data\Ajia_plc_1.csv
data\device_13_11_meter_1311.csv
Merged files with prefix "device_13_11_meter_1311" into data\device_13_11_meter_1311.csv
data\device_13_14_meter_1314.csv
Merged files with prefix "device_13_14_meter_1314" into data\device_13_14_meter_1314.csv
data\device_13_2_meter_1302.csv
Merged files with prefix "device_13_2_meter_1302" into data\device_13_2_meter_1302.csv
data\device_13_3_meter_1303.csv
Merged files with prefix "device_13_3_meter_1303" into data\device_13_3_meter_1303.csv
data\device_1_15_meter_115.csv
Merged files with prefix "device_1_15_meter_115" into data\device_1_15_meter_115.csv
data\device_1_2_meter_102.csv
Merged files with prefix "device_1_2_meter_102" into data\device_1_2_meter_102.csv
data\device_1_3_meter_103.csv
Merged files with prefix "device_1_3_meter_103" into data\device_1_3_meter_103.csv
data\device_1_5_meter_105.csv
Merged files with prefix "device_1_5_meter_105" into data\devic

In [2]:
def convert_to_numeric(value):
    try:
        return float(value)
    except ValueError:
        return -1

In [3]:
# 处理A架开关机动作
df = pd.read_csv('data/Ajia_plc_1.csv')
df['Ajia-3_v'] = df['Ajia-3_v'].apply(convert_to_numeric)
df['Ajia-5_v'] = df['Ajia-5_v'].apply(convert_to_numeric)
df['status'] = 'None' # 用于记录设备关键动作
df['is_device_powered'] = 'None' # 用于记录设备是否有电流，表示设备是否正在运行
for i in range(1, df.shape[0]):
    # 开机条件
    if df.loc[i - 1, 'Ajia-3_v'] == -1 and (df.loc[i, 'Ajia-3_v'] >= 0) and df.loc[i - 1, 'Ajia-5_v'] == -1 and (df.loc[i, 'Ajia-5_v'] >= 0):
        df.loc[i, 'status'] = '开机'
    # 关机条件
    if (df.loc[i, 'Ajia-3_v'] == -1 and (df.loc[i - 1, 'Ajia-3_v'] >= 0)) or (df.loc[i, 'Ajia-5_v'] == -1 and (df.loc[i - 1, 'Ajia-5_v'] >= 0)):
        df.loc[i, 'status'] = '关机'
    if (df.loc[i, 'Ajia-5_v'] > 0 and df.loc[i - 1, 'Ajia-5_v'] <= 0) and (df.loc[i, 'Ajia-3_v'] > 0 and df.loc[i - 1, 'Ajia-3_v'] <= 0):
        df.loc[i, 'is_device_powered'] = '有电流'
    if (df.loc[i, 'Ajia-5_v'] == 0 and df.loc[i - 1, 'Ajia-5_v'] >= 0) and (df.loc[i, 'Ajia-3_v'] == 0 and df.loc[i - 1, 'Ajia-3_v'] >= 0):
        df.loc[i, 'is_device_powered'] = '无电流'

In [ ]:
# 处理A架电流数据
from datetime import datetime
import pandas as pd

start_time = None
segments = []

# 遍历DataFrame
for index, row in df.iterrows():
    if row['status'] == '开机':
        start_time = row['csvTime']
    elif row['status'] == '关机' and start_time is not None:
        end_time = row['csvTime']
        segments.append((start_time, end_time))
        start_time = None

import json
XIAFANG = """你非常细心，通过仔细给定观察序列数据电流变化，尽可能正确返回三个值。
例如：
电流变化序列数据：
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.6478, 56.5133, 60.8637, 56.3751, 56.3777, 56.3601, 61.1564, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 422.499, 56.2896, 66.3951, 60.8928, 57.7813, 56.3871, 66.3077, 62.5263, 56.3937, 58.0826, 90.0969, 87.5592, 83.9934, 56.5033, 59.3441, 58.0018, 56.3027, 56.2845, 56.3666, 101.763, 96.6118, 56.3492, 59.2629, 57.0112, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]
在这里面寻找三个值返回结果：
[90.0969,56.5033,101.763]
返回依据解释：
第一个值为 最后一段非零数据的第一次达到峰值，同时满足必须为80以上；
第二个值为 要小于60，峰值回落至小于60的值,并且要在第二个值后面，在第三个值前面；
第三个值为 重新达到峰值，同时满足必须为90以上。
三个值均不考虑大于200异常数据


现有一组新的电流变化序列数据：
<<L>>
请参照样例和解释依据，新的电流变化序列数据可能会有一些噪声，你自己仔细思考判断，思考完成后，不需要返回思考过程，以列表形式返回三个值,回答中只有列表。。"""

HUISHOU = """你非常细心，通过仔细给定观察序列数据电流变化，尽可能正确返回三个值。
例如：
电流变化序列数据：
[0.0, 0.0, 0.0, 0.0, 0.0, 57.0048, 56.8545, 61.9802, 56.8646, 56.8705, 56.777, 68.3751, 56.5526, 56.6556, 63.1736, 68.4542, 78.2151, 86.3214, 82.7017, 58.9111, 56.632, 56.9142, 56.6583, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.2542, 56.2177, 56.1263, 56.2697, 56.102, 59.5568, 57.5703, 57.6415, 56.9307, 57.0531, 56.9337, 58.582, 58.0159, 104.238, 96.6301, 97.1496, 56.5543, 63.426, 57.6552, 56.6086, 56.6611, 56.5601, 56.6476, 56.68, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]
在这里面寻找三个值返回结果：
[86.3214,104.238,56.5543]
返回依据解释：
第一个值为 非最后一段非零数据的峰值，同时满足必须为80以上；
第二个值为后面一段非零数据的峰值，一般要为90以上；
第三个值 要小于60，峰值回落至小于60的值,并且要在第二个值后面。
三个值均不考虑大于200异常数据


现有一组新的电流变化序列数据：
<<L>>
请参照样例和解释依据，新的电流变化序列数据可能会有一些噪声，你自己仔细思考判断，思考完成后，不需要返回思考过程，以列表形式返回三个值,,回答中只有列表。。"""


def predict_sequence_by_llm(L_sequence, oper):
    from api import simple_completion

    if oper == 0:  #
        context_text = XIAFANG  # 直接读取为字符串
    else:
        context_text = HUISHOU
    prompt = context_text.replace("<<L>>", L_sequence)
    print(prompt)
    messages = [{"role": "user", "content": prompt}]
    response = simple_completion(messages)
    return str(response.choices[0].message.content)


def get_result(L_sequence, oper):
    try:
        # 第一次尝试
        input_string = predict_sequence_by_llm(L_sequence=L_sequence, oper=oper)
        print(input_string)
        result_list = json.loads(input_string)

        if len(result_list) == 3:
            a = result_list[0]
            b = result_list[1]
            c = result_list[2]
            return a, b, c
    except:
        # 第一次失败，进行第二次尝试
        try:
            input_string = predict_sequence_by_llm(L_sequence=L_sequence)
            result_list = json.loads(input_string)
            if len(result_list) == 3:
                a = result_list[0]
                b = result_list[1]
                c = result_list[2]
                return a, b, c
        except:
            # 第二次也失败，返回 -100, -100, -100
            return -100, -100, -100


def extract_daily_power_on_times(df):
    """
    从CSV文件中提取一天内有两次开机的第一次和第二次开机时间。

    参数:
    file_path (str): CSV文件的路径，包含 'csvTime' 和 'status' 列。

    返回:
    first_start_times (list): 一天内有两次开机的第一次开机时间列表。
    second_start_times (list): 一天内有两次开机的第二次开机时间列表。
    """
    # 读取CSV文件
    df = df

    # 将 csvTime 转换为 datetime 类型
    df['csvTime'] = pd.to_datetime(df['csvTime'])

    # 按天分组
    df['date'] = df['csvTime'].dt.date

    # 初始化一个字典来存储每天的开机关机时间段
    daily_segments = {}

    # 遍历每一天的数据
    for date, group in df.groupby('date'):
        segments = []
        start_time = None

        # 遍历每一天的记录
        for index, row in group.iterrows():
            if row['status'] == '开机':
                start_time = row['csvTime']
            elif row['status'] == '关机' and start_time is not None:
                end_time = row['csvTime']
                segments.append((start_time, end_time))
                start_time = None

        # 将每天的开机关机时间段存入字典
        daily_segments[date] = segments

    # 统计每天的开机关机次数
    daily_counts = {date: len(segments) for date, segments in daily_segments.items()}

    # 筛选出一天内有两次开机关机的情况
    two_times_days = [date for date, count in daily_counts.items() if count == 2]

    # 初始化两个列表来存储第一次和第二次的开机时间
    first_start_times = []
    second_start_times = []

    # 遍历这些日期，提取第一次和第二次的开机时间
    for date in two_times_days:
        segments = daily_segments[date]
        first_start_times.append(segments[0][0])  # 第一次开机时间
        second_start_times.append(segments[1][0])  # 第二次开机时间

    return first_start_times, second_start_times


def find_peaks(input_data):
    """
    找到列表中的峰值
    
    返回:
    tuple: 峰值的数量和峰值列表。
    """
    data = [50 if 50 <= num <= 68 else num for num in input_data]
    peaks = []
    for i in range(1, len(data) - 1):
        if data[i] > data[i - 1] and data[i] > data[i + 1]:
            peaks.append(data[i])
    peaks = [peak for peak in peaks if peak > 80]
    return len(peaks), peaks


def find_first_increasing_value(input_data):
    """
    找到列表中第一个从稳定值（68以下）开始增加的值。

    返回:
    tuple: 第一个大于68的值。如果未找到，返回50。
    """
    processed_data = [50 if 50 <= num <= 68 else num for num in input_data]
    for index, value in enumerate(processed_data):
        if value > 68 and value < 300:
            return value
    return 50


def find_first_stable_value_between_peaks(data, peak1, peak2, stable_min=50, stable_max=60):
    """
    找到两个峰值之间的数据中，回落到稳定值的第一个值。

    参数:
    data (list): 数据列表
    peak1 (float): 第一个峰值
    peak2 (float): 第二个峰值
    stable_min (float): 稳定值的最小值
    stable_max (float): 稳定值的最大值

    返回:
    float or None: 稳定值，如果未找到则返回 None
    """
    try:
        start_index = data.index(peak1)
        end_index = data.index(peak2)
    except ValueError:
        return None

    between_peaks = data[start_index:end_index + 1]

    for value in between_peaks:
        if stable_min <= value <= stable_max:
            return value
    return None


def find_first_stable_value_after_peak(data, peak, stable_min=50, stable_max=60):
    """
    从峰值到列表末尾的数据中，找到第一个回落到稳定值的值。

    参数:
    data (list): 数据列表
    peak (float): 峰值
    stable_min (float): 稳定值的最小值
    stable_max (float): 稳定值的最大值

    返回:
    float or None: 稳定值，如果未找到则返回 None
    """
    try:
        start_index = data.index(peak)
    except ValueError:
        return None

    after_peak = data[start_index:]

    for value in after_peak:
        if stable_min <= value <= stable_max:
            return value

    return None


def extract_events(df, segment):
    """
    从数据中提取事件，每个事件表示从"有电流"到"无电流"的过程
    
    参数:
    df (DataFrame): 包含 'csvTime' 和 'is_device_powered' 列的DataFrame。
    segment (tuple): 一个包含开机和关机时间的元组。
    
    返回:
    list: 事件列表，每个元素表示一个事件的峰值数量。
    """
    start, end = segment
    events = df[
        (df['csvTime'] >= start) & (df['csvTime'] <= end) & (df['is_device_powered'].isin(['有电流', '无电流']))]
    result = []
    if events.shape[0] >= 2 and events.shape[0] % 2 == 0:
        for i in range(0, events.shape[0], 2):
            event_start = events.iloc[i]
            event_end = events.iloc[i + 1]
            # 必须先有电流，再无电流
            if event_start['is_device_powered'] == '有电流' and event_end['is_device_powered'] == '无电流':
                start_event_time = event_start['csvTime']
                end_event_time = event_end['csvTime']
                between_events = df[(df['csvTime'] >= start_event_time) & (df['csvTime'] <= end_event_time)]
                len_peaks, peak_list = find_peaks(list(between_events['Ajia-5_v']))
                print("峰值为",peak_list)
                result.append(len_peaks)
    return result


events_in_all_segements = []
# 提取每个区段内的“通电流”和“关电流”事件
for segment in segments:
    events_in_segment = extract_events(df, segment=segment)
    events_in_all_segements.append(events_in_segment)
    start, end = segment
    events = df[(df['csvTime'] >= start) & (df['csvTime'] <= end) & (df['is_device_powered'].isin(['有电流', '无电流']))]
    events_2 = df[(df['csvTime'] >= start) & (df['csvTime'] <= end)]
    if events_in_segment in ([0, 2], [2], [0, 3], [0, 1, 3]):
        num1=(len(events_in_segment)-1)*2
        num2=events_in_segment[-1]-2
        if events.shape[0] >= 2 and events.shape[0] % 2 == 0:
            if events.iloc[num1+0]['is_device_powered'] == '有电流' and events.iloc[num1+1][
                'is_device_powered'] == '无电流':
                start_event_time = events.iloc[num1+0]['csvTime']
                end_event_time = events.iloc[num1+1]['csvTime']
                between_events = df[(df['csvTime'] >= start_event_time) & (df['csvTime'] <= end_event_time)]
                between_events_datalist = list(between_events['Ajia-5_v'])

                len_peaks, peak_list = find_peaks(between_events_datalist)

                if len_peaks == 0:
                    raise Exception("[0,2]未找到峰值")
                else:
                    value_11 = find_first_increasing_value(between_events_datalist)
                    indices = between_events.index[between_events['Ajia-5_v'] == value_11].tolist()
                    df.loc[indices, 'status'] = '征服者起吊'

                    value_11 = find_first_stable_value_between_peaks(between_events_datalist, peak_list[num2+0], peak_list[num2+1])
                    indices = between_events.index[between_events['Ajia-5_v'] == value_11].tolist()
                    df.loc[indices, 'status'] = '缆绳解除'
                    previous_indices = [idx - 1 for idx in indices if idx > 0]
                    df.loc[previous_indices, 'status'] = '征服者入水'

                    # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                    indices = between_events.index[between_events['Ajia-5_v'] == peak_list[num2+1]].tolist()
                    df.loc[indices, 'status'] = 'A架摆回'
    elif events_in_segment == [1, 2] or events_in_segment == [1, 1]:
        if events.shape[0] % 2 == 0:
            if events.iloc[0]['is_device_powered'] == '有电流' and events.iloc[1][
                'is_device_powered'] == '无电流':
                start_event_time = events.iloc[0]['csvTime']
                end_event_time = events.iloc[1]['csvTime']
                between_events = df[(df['csvTime'] >= start_event_time) & (df['csvTime'] <= end_event_time)]
                between_events_datalist = list(between_events['Ajia-5_v'])

                len_peaks, peak_list = find_peaks(between_events_datalist)
                value_11 = find_first_increasing_value(between_events_datalist)
                indices = between_events.index[between_events['Ajia-5_v'] == peak_list[0]].tolist()
                df.loc[indices, 'status'] = 'A架摆出'
                if events.iloc[2]['is_device_powered'] == '有电流' and events.iloc[3][
                    'is_device_powered'] == '无电流':
                    start_event_time = events.iloc[2]['csvTime']
                    end_event_time = events.iloc[3]['csvTime']
                    between_events = df[(df['csvTime'] >= start_event_time) & (df['csvTime'] <= end_event_time)]
                    between_events_datalist = list(between_events['Ajia-5_v'])

                    len_peaks, peak_list = find_peaks(between_events_datalist)
                    max_value = max([x for x in between_events_datalist if x <= 200])
                    # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                    indices = between_events.index[between_events['Ajia-5_v'] == max_value].tolist()
                    df.loc[indices, 'status'] = '征服者出水'
                    previous_indices = [idx - 1 for idx in indices if idx > 0]
                    df.loc[previous_indices, 'status'] = '缆绳挂妥'

                    value_11 = find_first_stable_value_after_peak(between_events_datalist, max_value)
                    indices = between_events.index[between_events['Ajia-5_v'] == value_11].tolist()
                    df.loc[indices, 'status'] = '征服者落座'                 
    else:
        events_2 = events_2.copy()
        events_2.loc[:, 'csvTime'] = pd.to_datetime(events_2['csvTime'])
        # 获取第一个值
        first_value = events_2['csvTime'].iloc[0]

        # 定义目标日期
        target_date = datetime(2024, 8, 19)  # 2024年8月19日   大模型预测测试
        is_target_date = (first_value.date() == target_date.date())

        target_date_1 = datetime(2024, 8, 19)  # 2024年8月19日   大模型预测测试
        is_target_date_1 = (first_value.date() == target_date.date())

        # 判断小时是否大于12点
        is_hour_greater_than_12 = first_value.hour > 17
        first_start_times, second_start_times = extract_daily_power_on_times(df=df)

        if is_target_date and is_hour_greater_than_12:  # 全部预测可以去掉is_target_date条件 或者根据问题传入
            events_2['new_column'] = events_2.apply(
                lambda row: row['Ajia-3_v'] if row['Ajia-5_v'] == 0 and row['Ajia-3_v'] > 0 else row['Ajia-5_v'],
                axis=1
            )
            print('----------------LLM预测的列表---------------------')
            print(str(list(events_2['new_column'])))

            try:
                a, b, c = get_result(str(list(events_2['new_column'])), 1)
            except Exception as e:
                print(f"An error occurred: {e}")
                a, b, c = -100, -100, -100
            print('----------------预测的值---------------------')
            print(a, b, c)

            indices = events_2.index[events_2['new_column'] == a].tolist()
            df.loc[indices, 'status'] = 'A架摆出'

            indices = events_2.index[events_2['new_column'] == b].tolist()
            df.loc[indices, 'status'] = '征服者出水'
            previous_indices = [idx - 1 for idx in indices if idx > 0]
            df.loc[previous_indices, 'status'] = '缆绳挂妥'

            indices = events_2.index[events_2['new_column'] == c].tolist()
            df.loc[indices, 'status'] = '征服者落座'



        elif first_value in first_start_times and 1 == 0:  # 去掉1==0由LLM判断状态  默认不开启 给大家分享参考思路。
            events_2['new_column'] = events_2.apply(
                lambda row: row['Ajia-3_v'] if row['Ajia-5_v'] == 0 and row['Ajia-3_v'] > 0 else row['Ajia-5_v'],
                axis=1
            )
            print('----------------LLM预测的列表---------------------')
            print(str(list(events_2['new_column'])))

            try:
                a, b, c = get_result(str(list(events_2['new_column'])), 0)
            except Exception as e:
                print(f"An error occurred: {e}")
                a, b, c = -100, -100, -100
            print('----------------预测的值---------------------')
            print(a, b, c)

            indices = events_2.index[events_2['new_column'] == a].tolist()
            df.loc[indices, 'status'] = '征服者起吊'

            indices = events_2.index[events_2['new_column'] == b].tolist()
            df.loc[indices, 'status'] = '缆绳解除'
            previous_indices = [idx - 1 for idx in indices if idx > 0]
            df.loc[previous_indices, 'status'] = '征服者入水'

            indices = events_2.index[events_2['new_column'] == c].tolist()
            df.loc[indices, 'status'] = 'A架摆回'

        elif first_value in second_start_times and 1 == 0:  # 去掉1==0由LLM判断状态
            events_2['new_column'] = events_2.apply(
                lambda row: row['Ajia-3_v'] if row['Ajia-5_v'] == 0 and row['Ajia-3_v'] > 0 else row['Ajia-5_v'],
                axis=1
            )
            print('----------------LLM预测的列表---------------------')
            print(str(list(events_2['new_column'])))

            try:
                a, b, c = get_result(str(list(events_2['new_column'])), 1)
            except Exception as e:
                print(f"An error occurred: {e}")
                a, b, c = -100, -100, -100
            print('----------------预测的值---------------------')
            print(a, b, c)

            indices = events_2.index[events_2['new_column'] == a].tolist()
            df.loc[indices, 'status'] = 'A架摆出'

            indices = events_2.index[events_2['new_column'] == b].tolist()
            df.loc[indices, 'status'] = '征服者出水'
            previous_indices = [idx - 1 for idx in indices if idx > 0]
            df.loc[previous_indices, 'status'] = '缆绳挂妥'

            indices = events_2.index[events_2['new_column'] == c].tolist()
            df.loc[indices, 'status'] = '征服者落座'

        print('------------------')
        print(events_in_segment)
df = df.drop(columns=['date'])  
df = df.drop(columns=['is_device_powered'])  
df.to_csv('database_in_use/Ajia_plc_1.csv', index=False)

In [ ]:
# 处理ON DP和OFF DP动作
import pandas as pd

df = pd.read_csv('data/Port3_ksbg_9.csv')
df['P3_33'] = pd.to_numeric(df['P3_33'], errors='coerce')
df['status'] = 'None' # 用于记录设备关键动作
for i in range(1, df.shape[0]):
    if df.loc[i - 1, 'P3_33'] == 0 and df.loc[i, 'P3_33'] > 0:
        df.loc[i, 'status'] = 'ON_DP'
    if df.loc[i - 1, 'P3_33'] > 0 and df.loc[i, 'P3_33'] == 0:
        df.loc[i, 'status'] = 'OFF_DP'
df.to_csv('database_in_use/Port3_ksbg_9.csv', index=False)

In [ ]:
# 处理折臂吊车和小艇动作
import pandas as pd
from collections import Counter

df = pd.read_csv('data/device_13_11_meter_1311.csv')
df['13-11-6_v'] = pd.to_numeric(df['13-11-6_v'], errors='coerce')
df['status'] = 'None' # 用于记录设备关键动作
df['action'] = 'None' # 用于记录设备关键动作

def sliding_window_5(arr):
    """
    滑动窗口大小为5的逻辑
    """
    window_size = 5
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i:i + window_size]
        if window[1] < 10 and window[2] < 10 and window[3] < 10 and window[0] > 10 and window[4] > 10:
            # 将 window[0] 包装成列表进行赋值
            modified_arr[i + 1:i + 4] = [window[0]] * 3
    return modified_arr


def sliding_window_4(arr):
    """
    滑动窗口大小为4的逻辑
    """
    window_size = 4
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i:i + window_size]
        if window[1] < 10 and window[2] < 10 and window[0] > 10 and window[3] > 10:
            # 将 window[0] 包装成列表进行赋值
            modified_arr[i + 1:i + 3] = [window[0]] * 2
    return modified_arr


def sliding_window_3(arr):
    """
    滑动窗口大小为3的逻辑
    """
    window_size = 3
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i:i + window_size]
        if window[1] < 10 and window[0] > 10 and window[2] > 10:
            # 直接赋值，因为只修改一个值
            modified_arr[i + 1] = window[0]
    return modified_arr


# 应用滑动窗口逻辑到 DataFrame 的某一列
df['13-11-6_v_new'] = sliding_window_5(df['13-11-6_v'].tolist())
df['13-11-6_v_new'] = sliding_window_4(df['13-11-6_v_new'].tolist())
df['13-11-6_v_new'] = sliding_window_3(df['13-11-6_v_new'].tolist())

# 检测折臂吊车的开机和关机事件
segments = []
start_time = None

for i in range(1, df.shape[0]):
    # 开机——数值从0增加（取增加）
    if df.iloc[i - 1]['13-11-6_v'] == 0 and df.iloc[i]['13-11-6_v'] > 0:
        df.at[df.index[i], 'status'] = '折臂吊车开机'
    # 关机——数值归零（取0）
    if df.iloc[i - 1]['13-11-6_v'] > 0 and df.iloc[i]['13-11-6_v'] == 0:
        df.at[df.index[i], 'status'] = '折臂吊车关机'
    # 检测由待机进入工作和由工作进入待机的事件
    if df.iloc[i - 1]['13-11-6_v_new'] < 10 and df.iloc[i]['13-11-6_v_new'] > 10:
        df.at[df.index[i], 'action'] = '由待机进入工作'
    if df.iloc[i - 1]['13-11-6_v_new'] > 10 and df.iloc[i]['13-11-6_v_new'] < 10:
        df.at[df.index[i], 'action'] = '由工作进入待机'
for index, row in df.iterrows():
    if row['status'] == '折臂吊车开机':
        start_time = row['csvTime']
    elif row['status'] == '折臂吊车关机' and start_time is not None:
        end_time = row['csvTime']
        segments.append((start_time, end_time))
        start_time = None


def find_most_frequent_number(lst):
    # 使用 Counter 统计每个数的出现次数
    counter = Counter(lst)
    # 找到出现次数最多的数（如果有多个，只返回第一个）
    most_common_number = counter.most_common(1)[0][0]
    return most_common_number


# 提取每个区段内的“由待机进入工作”和“由工作进入待机”事件
for segment in segments:
    start, end = segment
    events = df[
        (df['csvTime'] >= start) & (df['csvTime'] <= end) & (df['action'].isin(['由待机进入工作', '由工作进入待机']))]
    events_2 = df[(df['csvTime'] >= start) & (df['csvTime'] <= end)]
    # 检查事件数量是否为偶数且等于6
    if events.shape[0] == 6:
        print(f"开机时间: {start}, 关机时间: {end}")
        print(f"事件数量: {events.shape[0]}")
        # 处理每一对事件
        for i in range(0, 6, 2):

            event_start = events.iloc[i]
            event_end = events.iloc[i + 1]

            if event_start['action'] == '由待机进入工作' and event_end['action'] == '由工作进入待机':
                start_event_time = event_start['csvTime']
                end_event_time = event_end['csvTime']
                between_events = df[(df['csvTime'] >= start_event_time) & (df['csvTime'] <= end_event_time)]
                input_data = list(between_events['13-11-6_v'])

                # 找到最后一个大于9的值
                last_value_above_9 = next((x for x in reversed(input_data) if x > 9), None)

                if last_value_above_9 is not None:
                    all_indices = between_events.index[between_events['13-11-6_v_new'] == last_value_above_9].tolist()
                    last_index = all_indices[-1] if all_indices else None

                    # 根据事件对的顺序更新status
                    if last_index is not None:
                        if i == 0:
                            df.loc[last_index, 'status'] = '小艇检查完毕'
                        elif i == 2:
                            df.loc[last_index, 'status'] = '小艇入水'
                        elif i == 4:
                            df.loc[last_index, 'status'] = '小艇落座'
                else:
                    print("列表中没有大于 9 的值")
    if events.shape[0] == 4:
        print(f"开机时间: {start}, 关机时间: {end}")
        print(f"事件数量: {events.shape[0]}")
        # 处理每一对事件
        for i in range(0, 4, 2):
            event_start = events.iloc[i]
            event_end = events.iloc[i + 1]
            if event_start['action'] == '由待机进入工作' and event_end['action'] == '由工作进入待机':
                start_event_time = event_start['csvTime']
                end_event_time = event_end['csvTime']
                between_events = df[(df['csvTime'] >= start_event_time) & (df['csvTime'] <= end_event_time)]
                input_data = list(between_events['13-11-6_v'])

                # 找到最后一个大于9的值
                last_value_above_9 = next((x for x in reversed(input_data) if x > 9), None)

                if last_value_above_9 is not None:
                    all_indices = between_events.index[between_events['13-11-6_v_new'] == last_value_above_9].tolist()
                    last_index = all_indices[-1] if all_indices else None

                    # 根据事件对的顺序更新status

                    if last_index is not None and df.loc[last_index, 'status'] == "FALSE":
                        if i == 0:
                            df.loc[last_index, 'status'] = '小艇入水'
                        elif i == 2:
                            df.loc[last_index, 'status'] = '小艇落座'
                else:
                    print("列表中没有大于 9 的值")
                # 保存结果
df = df.drop(columns=['action'])
df = df.drop(columns=['13-11-6_v_new'])

df.to_csv('database_in_use/device_13_11_meter_1311.csv', index=False)